In [1]:
import pandas as pd

# House Election Data

In [2]:
# Load house election result data 1976-2022
file_path = "/Users/bill_zyx/jupyter-1.0.0/Trust/1976-2022-house.csv"
df_house = pd.read_csv(file_path)
df_house = df_house.rename(columns={'candidatevotes': 'candidate_votes', 'totalvotes':'total_votes'})

In [3]:
# Cleaning
columns_to_drop = ['state','state_fips','state_cen','state_ic','office','stage','runoff','mode','writein','version','special','candidate','unofficial','fusion_ticket']
df_house = df_house.drop(columns=columns_to_drop)   #Drop non-useful columns
df_house = df_house[df_house['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]   #Drop non-major parties
df_house = df_house[df_house['total_votes'] > 0]   #Drop problematic rows

In [4]:
# Align district code for at-large states and DC
df_house.loc[df_house['district'] == 0, 'district'] = 1

In [5]:
# Find share of votes
df_house['vote_share'] = df_house['candidate_votes']/df_house['total_votes']

# Find winning party
df_house['winner'] = df_house.groupby(['year', 'state_po', 'district'])['vote_share'].transform(
    lambda x: (x == x.max()).astype(int)
)

# Define democrat winning margin for RDD
df_house['dem_share'] = df_house.apply(
    lambda row: row['candidate_votes'] / row['total_votes'] if row['party'] == 'DEMOCRAT' else None,
    axis=1
)
df_house['dem_share'] = df_house.groupby(['year', 'state_po', 'district'])['dem_share'].transform('max')
df_house['dem_win_margin'] = df_house['dem_share'] - 0.5   # win margin relative to 50%

In [6]:
# Drop losing rows
df_house = df_house[~df_house['winner'].isin([0])]
df_house = df_house.drop(columns='winner')

# Drop absolute columns (not useful for analysis)
df_house = df_house.drop(columns='candidate_votes')

# Drop NaN values (eg., some dem_win_margin missing due to original data deficiency)
df_house = df_house.dropna()

In [7]:
# Unify id variable format and name
df_house["district"] = (
    df_house["district"]
        .astype("Int64")        
        .astype("string")
        .str.zfill(2)
)

df_house = df_house.rename(
    columns={
        'state_po': 'state_code',
        'district': 'district_code'
    }
)

In [8]:
# Export cleaned house election data
df_house.to_csv("data/intermediate/house_election_district_level.csv")